In [7]:
import os
import ast
import json
import time
import shutil
import zipfile
import warnings

warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import rasterio
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

print(f'PyTorch version: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
if torch.cuda.is_available():
    device = torch.device('cuda')
elif torch.backends.mps.is_available():
    device = torch.device('mps')
else:
    device = torch.device('cpu')
print(f'Device: {device}')

PyTorch version: 2.12.1
CUDA available: False
Device: mps


In [1]:
import os
from pathlib import Path

# --- Locate Workspace and Verify Extracted Datasets ---
# Works whether the kernel starts in the workspace root or in src/.
candidates = [Path.cwd(), Path.cwd().parent]
workspace_root = next(
    (path for path in candidates
     if (path / 'train_dataset').is_dir()
     and (path / 'evaluation_dataset').is_dir()),
    None,
)
if workspace_root is None:
    raise FileNotFoundError('train_dataset/ and evaluation_dataset/ were not found.')

os.chdir(workspace_root)
print(f'  ✓ Workspace: {workspace_root}')
print('  ✓ train_dataset/ found')
print('  ✓ evaluation_dataset/ found')

  ✓ Workspace: /Users/shionsuio/solafune-workspace
  ✓ train_dataset/ found
  ✓ evaluation_dataset/ found


In [3]:
# --- Configuration ---
# Paths relative to this notebook. Adjust if your directory structure differs.

TRAIN_DIR = 'train_dataset'
TEST_DIR = 'evaluation_dataset'

TRAIN_CSV = os.path.join(TRAIN_DIR, 'train_dataset.csv')
EVAL_TARGET_CSV = os.path.join(TEST_DIR, 'evaluation_target.csv')

# Satellite data folders
SATELLITE_DIRS = {
    'himawari': {'train': os.path.join(TRAIN_DIR, 'himawari'), 'test': os.path.join(TEST_DIR, 'himawari')},
    'meteosat': {'train': os.path.join(TRAIN_DIR, 'meteosat'), 'test': os.path.join(TEST_DIR, 'meteosat')},
    'goes':     {'train': os.path.join(TRAIN_DIR, 'goes'),     'test': os.path.join(TEST_DIR, 'goes')},
}

# Target data folder (training only — GPM IMERG precipitation maps)
GPM_TRAIN_DIR = os.path.join(TRAIN_DIR, 'gpm_imerg')

# Output directories
MODEL_DIR = 'models'
os.makedirs(MODEL_DIR, exist_ok=True)

# Verify paths
for name, path in [('Train CSV', TRAIN_CSV), ('Eval CSV', EVAL_TARGET_CSV), ('GPM Train', GPM_TRAIN_DIR)]:
    status = '✓' if os.path.exists(path) else '✗'
    print(f'  {status} {name}')
for sat_name, dirs in SATELLITE_DIRS.items():
    for split, path in dirs.items():
        status = '✓' if os.path.exists(path) else '✗'
        print(f'  {status} {sat_name}/{split}')

  ✓ Train CSV
  ✓ Eval CSV
  ✓ GPM Train
  ✓ himawari/train
  ✓ himawari/test
  ✓ meteosat/train
  ✓ meteosat/test
  ✓ goes/train
  ✓ goes/test


In [5]:
# Hyperparameters
TARGET_SIZE = (41, 41)
ENCODE_SIZE = (224, 224) #Swin-Tの入力解像度
NUM_BANDS = 16 #全バンド使用
MAX_OBS = 3
INPUT_CHANNELS = NUM_BANDS * MAX_OBS  # 48
OUTPUT_CHANNELS = 1
BATCH_SIZE = 8 #Swin-Tはメモリが重いので下げる
LEARNING_RATE = 2e-4
EPOCHS = 30
WIGHT_DECAY = 1e-4 #過学習抑止

In [8]:
# --- Load Training Metadata ---

train_df = pd.read_csv(TRAIN_CSV)
print(f'Training samples: {len(train_df):,}')
print(f'Columns: {list(train_df.columns)}')
print(f'\nUnique locations: {sorted(train_df["name_location"].unique())}')
print(f'Unique satellites: {sorted(train_df["satellite_target"].unique())}')
print(f'\nSamples per satellite:')
print(train_df.groupby('satellite_target').size())
train_df.head()

Training samples: 40,686
Columns: ['unique_id', 'name_location', 'satellite_target', 'datetime', 'last_30_minutes_observation_filename', 'gpm_imerg_filename']

Unique locations: ['aceh', 'andalusia', 'atlantic_coast', 'bahia_blanca', 'bihar', 'borno_state', 'cape_town', 'central_philippines', 'central_vietnam', 'dhaka', 'ecuador', 'florida', 'france', 'friuli_venezia_giulia', 'gaza_province', 'guangdong', 'hat_yai', 'jakarta', 'jamaica', 'kinshasa']
Unique satellites: ['goes', 'himawari', 'meteosat']

Samples per satellite:
satellite_target
goes        10272
himawari    13192
meteosat    17222
dtype: int64


,unique_id,name_location,satellite_target,datetime,last_30_minutes_observation_filename,gpm_imerg_filename
0,c901-207d,aceh,himawari,2023-01-01 00:00:00,"['train_aceh_Himawari_20221231_2330.tif', 'tra...",train_aceh_GPM_IMERG_2023-01-01_00-00-00.tif
1,3311-9497,aceh,himawari,2023-01-01 00:30:00,"['train_aceh_Himawari_20230101_0000.tif', 'tra...",train_aceh_GPM_IMERG_2023-01-01_00-30-00.tif
2,ae46-e82d,aceh,himawari,2023-01-01 01:00:00,"['train_aceh_Himawari_20230101_0030.tif', 'tra...",train_aceh_GPM_IMERG_2023-01-01_01-00-00.tif
3,6ca6-b8d8,aceh,himawari,2023-01-01 01:30:00,"['train_aceh_Himawari_20230101_0100.tif', 'tra...",train_aceh_GPM_IMERG_2023-01-01_01-30-00.tif
4,20a1-eba0,aceh,himawari,2023-01-01 02:00:00,"['train_aceh_Himawari_20230101_0130.tif', 'tra...",train_aceh_GPM_IMERG_2023-01-01_02-00-00.tif


In [9]:
# --- Inspect Satellite TIF Properties ---

import glob

for sat_name in SATELLITE_DIRS:
    sat_dir = SATELLITE_DIRS[sat_name]['train']
    tif_files = sorted(glob.glob(os.path.join(sat_dir, '*.tif')))
    if not tif_files:
        print(f'{sat_name.upper()}: no TIF files found')
        continue
    with rasterio.open(tif_files[0]) as src:
        data = src.read()
        print(f'{sat_name.upper()}: {src.count} bands, {src.width}×{src.height}, '
              f'dtype={src.dtypes[0]}, bands 1-16 range=[{data[:16].min()}, {data[:16].max()}]')

HIMAWARI: 16 bands, 81×81, dtype=uint8, bands 1-16 range=[0, 221]
METEOSAT: 16 bands, 144×144, dtype=uint8, bands 1-16 range=[0, 225]
GOES: 16 bands, 141×141, dtype=uint8, bands 1-16 range=[0, 253]


In [10]:
# --- Inspect GPM IMERG Target ---

gpm_files = sorted(glob.glob(os.path.join(GPM_TRAIN_DIR, '*.tif')))
print(f'Total GPM IMERG training files: {len(gpm_files):,}')

with rasterio.open(gpm_files[0]) as src:
    gpm_data = src.read(1)
    print(f'GPM IMERG: {src.count} band, {src.width}×{src.height}, dtype={src.dtypes[0]}')
    print(f'  Value range: [{gpm_data.min():.4f}, {gpm_data.max():.4f}], '
          f'mean={gpm_data.mean():.4f}')

Total GPM IMERG training files: 40,686
GPM IMERG: 1 band, 41×41, dtype=float32
  Value range: [0.0260, 0.8530], mean=0.3633
